<a href="https://colab.research.google.com/github/nivethithanm/torchcode-solutions/blob/main/P0_08a_stable_cross_entropy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FIX-02: Stable Cross-Entropy — From First Principles

**Why you're stuck**: Cross-entropy combines 3 operations (softmax, log, indexing). When thrown together, it's confusing.

**This notebook**: We build each piece separately, then combine.

**Prerequisites**: You can do softmax (P0-04) and you know what gather does (FIX-01).

In [1]:
import torch
import torch.nn.functional as F

## Step 1: What cross-entropy loss IS

Classification: model outputs raw scores ('logits') for each class.
We want: high logit for the correct class, low logits for wrong classes.

```
Cross-entropy = -log(probability of correct class)
```

If model gives 90% probability to correct class: loss = -log(0.9) = 0.105 (small, good)
If model gives 10% probability to correct class: loss = -log(0.1) = 2.303 (big, bad)

In [2]:
import math
print('Correct class probability → Loss:')
for p in [0.9, 0.5, 0.1, 0.01]:
    print(f'  p = {p:.2f} → loss = {-math.log(p):.3f}')

Correct class probability → Loss:
  p = 0.90 → loss = 0.105
  p = 0.50 → loss = 0.693
  p = 0.10 → loss = 2.303
  p = 0.01 → loss = 4.605


## Step 2: From logits to probabilities (softmax)

In [3]:
# Model outputs raw logits (can be any real number)
logits = torch.tensor([2.0, 1.0, 0.5])  # 3 classes
correct_class = 0  # the answer is class 0

# Softmax converts to probabilities (0 to 1, sum to 1)
probs = torch.softmax(logits, dim=0)
print('Logits:', logits)
print('Probs: ', probs)
print('Sum:   ', probs.sum())  # 1.0

Logits: tensor([2.0000, 1.0000, 0.5000])
Probs:  tensor([0.6285, 0.2312, 0.1402])
Sum:    tensor(1.0000)


## Step 3: The NAIVE (unstable) cross-entropy

In [4]:
# Naive: softmax → log → pick correct class
logits = torch.tensor([2.0, 1.0, 0.5])
correct_class = 0

# Step by step:
probs = torch.softmax(logits, dim=0)       # [0.59, 0.22, 0.13]
log_probs = torch.log(probs)                 # [-0.52, -1.52, -2.02]
loss = -log_probs[correct_class]              # 0.52
print(f'Loss: {loss:.4f}')

# This WORKS for normal numbers. But what about extreme logits?

Loss: 0.4644


## Step 4: WHY it breaks (the stability problem)

In [5]:
# With large logits, exp() overflows
logits_big = torch.tensor([1000.0, 1001.0, 1002.0])

# exp(1000) = inf
probs_bad = torch.exp(logits_big) / torch.exp(logits_big).sum()
print('Naive probs:', probs_bad)  # [nan, nan, nan] — BROKEN

# Also: log(0) = -inf, log(negative) = nan
print('log(0):', torch.log(torch.tensor(0.0)))  # -inf

Naive probs: tensor([nan, nan, nan])
log(0): tensor(-inf)


## Step 5: The fix — log_softmax

Instead of computing `log(softmax(x))` as two steps, compute it as ONE step:

```
log_softmax(x_i) = x_i - log(sum(exp(x_j)))
                  = x_i - max(x) - log(sum(exp(x_j - max(x))))
```

By subtracting max(x) inside the exp, we prevent overflow.

In [6]:
# log_softmax is numerically stable
logits_big = torch.tensor([1000.0, 1001.0, 1002.0])

log_probs = F.log_softmax(logits_big, dim=0)
print('log_softmax:', log_probs)  # works! no nan/inf

# Let's implement it ourselves
def my_log_softmax(x, dim=-1):
    max_x = x.max(dim=dim, keepdim=True).values
    shifted = x - max_x  # subtract max for stability
    log_sum = torch.log(torch.exp(shifted).sum(dim=dim, keepdim=True))
    return shifted - log_sum

mine = my_log_softmax(logits_big, dim=0)
print('My version: ', mine)
assert torch.allclose(mine, log_probs, atol=1e-5)
print('✅ Matches!')

log_softmax: tensor([-2.4076, -1.4076, -0.4076])
My version:  tensor([-2.4076, -1.4076, -0.4076])
✅ Matches!


## Step 6: Putting it together — stable cross-entropy

Now combine log_softmax + gather:

In [7]:
def stable_cross_entropy(logits, targets):
    """Stable cross-entropy loss.
    Args:
        logits: (B, C) — raw model outputs
        targets: (B,) — correct class indices
    Returns: scalar loss
    """
    # Step 1: Compute log-probabilities (stable)
    log_probs = F.log_softmax(logits, dim=-1)  # (B, C)

    # Step 2: Pick the log-prob of the correct class for each sample
    # targets is (B,) → reshape to (B, 1) for gather
    targets_2d = targets.unsqueeze(1)  # (B, 1)
    correct_log_probs = torch.gather(log_probs, dim=1, index=targets_2d)  # (B, 1)

    # Step 3: Negate and average
    loss = -correct_log_probs.mean()

    return loss

# Test
logits = torch.randn(4, 10)
targets = torch.tensor([3, 7, 1, 5])

mine = stable_cross_entropy(logits, targets)
ref = F.cross_entropy(logits, targets)
print(f'Mine: {mine:.6f}')
print(f'Ref:  {ref:.6f}')
assert torch.allclose(mine, ref, atol=1e-5)
print('✅ Stable cross-entropy matches PyTorch!')

Mine: 3.079297
Ref:  3.079297
✅ Stable cross-entropy matches PyTorch!


## Exercise: Now YOU implement it from scratch

In [20]:
def my_cross_entropy(logits, targets):
    # TODO: Step 1 — log_softmax on logits along dim=-1
    logits_max = logits.max(dim=-1, keepdim=True)[0]
    logits_minus_max = logits - logits_max
    log_probs = logits_minus_max - torch.log(torch.sum(torch.exp(logits_minus_max), dim=-1, keepdim=True))  # YOUR CODE

    # TODO: Step 2 — gather correct class log-prob for each sample
    targets_2d = targets.unsqueeze(1)
    correct = log_probs.gather(dim=-1, index=targets_2d)  # YOUR CODE (remember: targets needs unsqueeze)

    # TODO: Step 3 — negate and average
    loss = -torch.mean(correct)  # YOUR CODE

    return loss

logits = torch.randn(8, 100)
targets = torch.randint(0, 100, (8,))
mine = my_cross_entropy(logits, targets)
ref = F.cross_entropy(logits, targets)
assert torch.allclose(mine, ref, atol=1e-5)
print('✅ You implemented cross-entropy from scratch!')

✅ You implemented cross-entropy from scratch!
